**Purpose:** `v5.2.X_reddit+news.ipynb` — part of `03_portfolio/LSTM_returns` (see the stage README).

**Outputs:** `LSTMvia-returns_reddit+news_weights.npy`, `test_bl_returns.csv`, `test_bl_weights.csv`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT
from src.seeds import SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_REDDIT_NEWS, SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_REDDIT_NEWS_BL


# LSTM + Black–Litterman — Reddit + News Merged Sentiment (v5.2.2_merged)

Extends v5.2.2_reddit and v5.2.2_news by **merging** both sentiment sources into
a single LSTM per rank.

### Pipeline
1. Load the **reddit** and **news** per-rank LSTM studies (already tuned in
   v5.2.2_reddit / v5.2.2_news).
2. For each rank `i`, extract the **frozen technical picks** (same for both,
   since they share the same v5.2.2 base) and the **frozen sentiment picks**
   from each source.
3. Build one combined feature vector per rank:
   `[technical_feats | reddit_sentiment_cols | news_sentiment_cols | _REDDIT_NAN | _NEWS_NAN]`
4. Run an independent Optuna study per rank to tune **only the LSTM + training
   hyperparameters** (features are fully frozen).
5. Run the BL search exactly as before (top_n fixed = len(studies)).
6. Final out-of-sample test with mid-test refit.

## 1) Imports

In [ ]:
from __future__ import annotations

import os
import json
import copy
import random
import logging
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

## 2) Config

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATASET_PATH  = str(PROJECT_ROOT / "03_portfolio/dataset.parquet")
SPREADS_PATH  = str(PROJECT_ROOT / "01_data/aux/bid-ask_spread.json")

# Paths to the per-rank LSTM study DBs produced by v5.2.2_reddit and v5.2.2_news.
# Adjust these to wherever those notebooks saved their output dirs.
# Expected layout:  {dir}/lstm_top{rank}/lstm_top{rank}.db
V522_REDDIT_OUT_DIR = "v522_reddit"   # parent dir that contains lstm_top0/, lstm_top1/, ...
V522_NEWS_OUT_DIR   = "v522_news"     # same structure

# ── Universe ──────────────────────────────────────────────────────────────
ASSETS = ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']

# ── Base technical feature buckets (carried from v5.2.2) ──────────────────
FEATURES = {
    'Price':      ['Close'],
    'OHLC':       ['Open', 'High', 'Low'],
    'Trend':      ['SMA5', 'SMA22', 'SMA63', 'SMA126', 'SMA252', 'MACD', 'ADX'],
    'Momentum':   ['RSI', 'MACD', 'CCI'],
    'Volume':     ['Volume', 'OBV', 'ADI', 'VolumeVariation'],
    'Volatility': ['BBUpper', 'BBLower', '5dayVolatility', '22dayVolatility',
                   '63dayVolatility', 'RatioVolatility'],
    'Market':     ['VIXIndexClose'],
}

# ── Study settings ────────────────────────────────────────────────────────
OUT_DIR      = 'v522_reddit+news'
N_TRIALS     = 0      # Optuna HP-only trials per merged LSTM rank
N_TRIALS_BL  = 0      # BL HPO trials
MODEL_SEEDS  = [11, 22, 33]
BL_SEED      = SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_REDDIT_NEWS_BL


## 3) Utilities

In [ ]:
def setup_logger(log_dir: str) -> logging.Logger:
    logger = logging.getLogger(f"merged_lstm_{os.path.basename(log_dir)}")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s")
    ch  = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    logger.propagate = False
    return logger


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def safe_makedirs(path: str):
    os.makedirs(path, exist_ok=True)

## 4) Walk-forward splits  *(unchanged)*

In [ ]:
def make_walk_forward_splits(dates_index: pd.DatetimeIndex):
    splits = []
    periods = [
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2020-06-30"),
         pd.Timestamp("2020-07-01"), pd.Timestamp("2021-06-30")),
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2021-06-30"),
         pd.Timestamp("2021-07-01"), pd.Timestamp("2022-06-30")),
        (pd.Timestamp("2015-07-01"), pd.Timestamp("2022-06-30"),
         pd.Timestamp("2022-07-01"), pd.Timestamp("2023-06-30")),
    ]
    for tr_start, tr_end, v_start, v_end in periods:
        train_idx = np.where((dates_index >= tr_start) & (dates_index <= tr_end))[0]
        val_idx   = np.where((dates_index >= v_start)  & (dates_index <= v_end))[0]
        if len(train_idx) == 0 or len(val_idx) == 0:
            print(f"[SKIP SPLIT] train {tr_start.date()}-{tr_end.date()} "
                  f"val {v_start.date()}-{v_end.date()} | "
                  f"lens train={len(train_idx)} val={len(val_idx)}")
            continue
        splits.append((train_idx, val_idx))
    if len(splits) == 0:
        raise ValueError("No valid walk-forward splits.")
    return splits


def split_train_for_early_stop(
    train_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    es_years: int = 1,
    min_train_days: int = 252,
) -> Tuple[np.ndarray, np.ndarray]:
    train_dates = dates_index[train_idx]
    train_end   = train_dates[-1]
    es_start    = train_end - pd.DateOffset(years=es_years) + pd.Timedelta(days=1)
    es_mask     = train_dates >= es_start
    core_idx    = train_idx[~es_mask]
    es_idx      = train_idx[es_mask]
    if len(core_idx) < min_train_days or len(es_idx) < min_train_days // 2:
        cut      = max(min_train_days, len(train_idx) - 252 * es_years)
        cut      = min(cut, len(train_idx) - 1)
        core_idx = train_idx[:cut]
        es_idx   = train_idx[cut:]
    if len(core_idx) == 0 or len(es_idx) == 0:
        raise ValueError("Failed to create non-empty early-stopping split.")
    return core_idx, es_idx

## 5) Load merged base configs from reddit & news studies

For each rank `i` we grab:
- **technical picks** (from reddit study — identical to news, both inherit v5.2.2)
- **reddit sentiment cols** (from reddit best trial)
- **news sentiment cols** (from news best trial)

The merged config bundles all of these; no new feature search happens here.

In [ ]:
def _load_rank_studies(parent_dir: str, top_n: int) -> List[optuna.Study]:
    """Load the per-rank LSTM studies from a v5.2.2_reddit or v5.2.2_news output dir."""
    studies = []
    for rank in range(top_n):
        db_path    = os.path.join(parent_dir, f"lstm_top{rank}", f"lstm_top{rank}.db")
        storage    = f"sqlite:///{db_path}"
        study_name = f"lstm_top{rank}"
        study = optuna.load_study(study_name=study_name, storage=storage)
        studies.append(study)
    return studies


def _get_top_n_from_reddit_studies(reddit_parent_dir: str) -> int:
    """
    Count how many lstm_top{rank} DBs exist under the reddit output dir.
    This is the TOP_N fixed by the v5.2.2 BL study.
    """
    rank = 0
    while os.path.exists(
        os.path.join(reddit_parent_dir, f"lstm_top{rank}", f"lstm_top{rank}.db")
    ):
        rank += 1
    if rank == 0:
        raise ValueError(f"No lstm_top* DBs found under {reddit_parent_dir!r}")
    print(f"  Found {rank} rank studies under {reddit_parent_dir!r}")
    return rank


def build_merged_configs(
    reddit_parent_dir: str,
    news_parent_dir: str,
) -> Tuple[List[Dict[str, Any]], int]:
    """
    For each rank i:
      - technical picks & per_asset_feats come from the reddit best trial
        (identical in news — both inherit the same v5.2.2 base)
      - reddit_sentiment_cols  come from the reddit best trial user_attrs
      - news_sentiment_cols    come from the news   best trial user_attrs
      - reddit_nan_strategy    from reddit best trial
      - news_nan_strategy      from news   best trial

    Returns:
      merged_configs : List[Dict] — one per rank
      top_n          : int
    """
    top_n = _get_top_n_from_reddit_studies(reddit_parent_dir)

    reddit_studies = _load_rank_studies(reddit_parent_dir, top_n)
    news_studies   = _load_rank_studies(news_parent_dir,   top_n)

    if len(reddit_studies) != len(news_studies):
        raise ValueError(
            f"Mismatch: {len(reddit_studies)} reddit ranks vs {len(news_studies)} news ranks"
        )

    merged_configs = []
    for rank, (r_study, n_study) in enumerate(zip(reddit_studies, news_studies)):
        r_best = r_study.best_trial
        n_best = n_study.best_trial

        # Technical features — identical between sources; take from reddit
        tech_picks       = r_best.user_attrs["frozen_picks"]
        per_asset_feats  = r_best.user_attrs["frozen_feats"]

        # Sentiment cols (suffixes like "reddit_threshold0.50_balance_ratio")
        r_sent_picks = r_best.user_attrs["sentiment_picks"]
        n_sent_picks = n_best.user_attrs["sentiment_picks"]

        reddit_sentiment_cols = r_sent_picks["sentiment_cols"]
        news_sentiment_cols   = n_sent_picks["sentiment_cols"]

        reddit_nan_strategy = r_best.user_attrs["reddit_nan_strategy"]
        news_nan_strategy   = n_best.user_attrs["news_nan_strategy"]

        # Merged sentiment = reddit cols + news cols (distinct prefixes, no collision)
        merged_sentiment_cols = reddit_sentiment_cols + news_sentiment_cols

        config = {
            "rank":                   rank,
            "tech_picks":             tech_picks,
            "per_asset_feats":        per_asset_feats,
            "reddit_sentiment_cols":  reddit_sentiment_cols,
            "news_sentiment_cols":    news_sentiment_cols,
            "merged_sentiment_cols":  merged_sentiment_cols,
            "reddit_nan_strategy":    reddit_nan_strategy,
            "news_nan_strategy":      news_nan_strategy,
            # Carry original trial metadata for logging
            "reddit_trial_number":    r_best.number,
            "reddit_ic":              r_best.value,
            "news_trial_number":      n_best.number,
            "news_ic":                n_best.value,
        }
        merged_configs.append(config)
        print(
            f"  Rank {rank}"
            f" | reddit trial #{r_best.number} IC={r_best.value:.5f}"
            f" | news trial #{n_best.number} IC={n_best.value:.5f}"
            f" | reddit_sent={reddit_sentiment_cols}"
            f" | news_sent={news_sentiment_cols}"
        )

    return merged_configs, top_n


# ── Run at notebook startup ───────────────────────────────────────────────
print("Building merged configs …")
MERGED_CONFIGS, TOP_N = build_merged_configs(
    reddit_parent_dir=V522_REDDIT_OUT_DIR,
    news_parent_dir=V522_NEWS_OUT_DIR,
)
print(f"\nLoaded {len(MERGED_CONFIGS)} merged configs (top_n={TOP_N}).")

## 6) Feature engineering helpers  *(unchanged)*

In [ ]:
def pick_features_from_buckets(trial: optuna.Trial, features: Dict[str, List[str]]) -> Dict[str, Any]:
    picks = {}
    picks["Price"] = "log_ret_1d"
    use_ohlc = trial.suggest_categorical("use_ohlc", [0, 1])
    picks["OHLC"] = ["Open", "High", "Low"] if use_ohlc == 1 else []
    for bucket in ["Trend", "Momentum", "Volume"]:
        picks[bucket] = trial.suggest_categorical(f"pick_{bucket}", features[bucket])
    vol_options = [f for f in features["Volatility"] if f not in ("BBLower", "BBUpper")]
    vol_options.append("BBands")
    vol_choice = trial.suggest_categorical("pick_Volatility", vol_options)
    picks["Volatility"] = ["BBLower", "BBUpper"] if vol_choice == "BBands" else vol_choice
    use_market = trial.suggest_categorical("use_market", [0, 1])
    picks["Market"] = "VIXIndexClose" if use_market == 1 else None
    return picks


def get_per_asset_feature_list(picks: Dict[str, Any]) -> List[str]:
    feats = ["log_ret_1d"]
    for f in picks.get("OHLC", []):
        feats.append(f)
    for bucket in ["Trend", "Momentum", "Volume", "Volatility"]:
        val = picks[bucket]
        if isinstance(val, list):
            feats.extend(val)
        else:
            feats.append(val)
    out  = []
    seen = set()
    for f in feats:
        if f not in seen:
            out.append(f)
            seen.add(f)
    return out


def required_df_columns_merged(
    assets: List[str],
    per_asset_feats: List[str],
    picks: Dict[str, Any],
    reddit_sentiment_cols: List[str],
    news_sentiment_cols: List[str],
) -> List[str]:
    """
    Build the full list of df columns needed for the merged model.
    Both _REDDIT_NAN and _NEWS_NAN are single global columns (not per-asset).
    """
    cols = []
    all_sentiment_cols = reddit_sentiment_cols + news_sentiment_cols
    for a in assets:
        cols.append(f"{a}_Close")
        for f in per_asset_feats:
            if f == "log_ret_1d":
                continue
            cols.append(f"{a}_{f}")
        for sf in all_sentiment_cols:
            cols.append(f"{a}_{sf}")
    # Global NaN flags
    if reddit_sentiment_cols:
        cols.append("_REDDIT_NAN")
    if news_sentiment_cols:
        cols.append("_NEWS_NAN")
    if picks.get("Market") is not None:
        cols.append(picks["Market"])
    return cols

## 7) Stationary transforms  *(extended for both sentiment sources)*

In [ ]:
def _winsorize_fit(x: np.ndarray, q: float) -> Tuple[float, float]:
    lo = np.nanquantile(x, q)
    hi = np.nanquantile(x, 1.0 - q)
    if not np.isfinite(lo): lo = 0.0
    if not np.isfinite(hi): hi = 0.0
    if hi < lo: hi = lo
    return float(lo), float(hi)


def _winsorize_apply(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    return np.clip(x, lo, hi)


@dataclass
class FeatureTransformSpec:
    kind: str
    winsor_q: float = 0.01
    clip_value: float = 8.0


@dataclass
class FittedTransform:
    spec: FeatureTransformSpec
    lo: float = 0.0
    hi: float = 0.0
    center: float = 0.0
    scale: float = 1.0

    def transform(self, x: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32)
        x = np.nan_to_num(x, nan=np.nan, posinf=np.nan, neginf=np.nan)
        if self.spec.kind in ("robust_z", "log1p_robust_z"):
            x = _winsorize_apply(x, self.lo, self.hi)
        if self.spec.kind == "identity":
            y = x
        elif self.spec.kind == "bounded_0_100":
            y = (x - 50.0) / 50.0
        elif self.spec.kind == "log1p_robust_z":
            y = np.sign(x) * np.log1p(np.abs(x))
            y = (y - self.center) / self.scale
        elif self.spec.kind == "robust_z":
            y = (x - self.center) / self.scale
        else:
            raise ValueError(f"Unknown transform kind: {self.spec.kind}")
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        y = np.clip(y, -self.spec.clip_value, self.spec.clip_value)
        return y.astype(np.float32)


def default_transform_spec(feature_name: str) -> FeatureTransformSpec:
    """Assign a normalisation strategy by feature name — handles both reddit & news prefixes."""
    name = feature_name.lower()
    # ── Technical ──────────────────────────────────────────────────────────
    if feature_name == "log_ret_1d":
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
    if name in ("rsi", "adx"):
        return FeatureTransformSpec(kind="bounded_0_100", clip_value=2.0)
    if name in ("volume", "volumevariation", "obv", "adi"):
        return FeatureTransformSpec(kind="log1p_robust_z", winsor_q=0.01, clip_value=8.0)
    if name in ("open", "high", "low"):
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
    # ── NaN flags (binary 0/1) ─────────────────────────────────────────────
    if name in ("_reddit_nan", "_news_nan", "reddit_nan", "news_nan"):
        return FeatureTransformSpec(kind="identity", clip_value=2.0)
    # ── Count-based sentiment (skewed) ────────────────────────────────────
    if "count_positive" in name or "count_negative" in name or name.endswith("_balance"):
        return FeatureTransformSpec(kind="log1p_robust_z", winsor_q=0.01, clip_value=8.0)
    # ── Ratio / balance_ratio variants ───────────────────────────────────
    if "ratio" in name or "variation" in name or "balance_ratio" in name:
        return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
    # ── Default ────────────────────────────────────────────────────────────
    return FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)


def fit_transforms_on_train(
    X_train_raw: np.ndarray,
    feat_names: List[str],
    market_train_raw: Optional[np.ndarray],
) -> Tuple[List[FittedTransform], Optional[FittedTransform]]:
    fitted = []
    for j, fname in enumerate(feat_names):
        spec = default_transform_spec(fname)
        vals = X_train_raw[:, :, j].reshape(-1)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            fitted.append(FittedTransform(spec=spec))
            continue
        if spec.kind in ("robust_z", "log1p_robust_z"):
            lo, hi  = _winsorize_fit(vals, spec.winsor_q)
            vals_w  = _winsorize_apply(vals, lo, hi)
            if spec.kind == "log1p_robust_z":
                vals_w = np.sign(vals_w) * np.log1p(np.abs(vals_w))
            center = float(np.nanmedian(vals_w))
            mad    = float(np.nanmedian(np.abs(vals_w - center)))
            scale  = 1.4826 * mad
            if not np.isfinite(scale) or scale < 1e-6:
                scale = float(np.nanstd(vals_w))
            if not np.isfinite(scale) or scale < 1e-6:
                scale = 1.0
            fitted.append(FittedTransform(spec=spec, lo=lo, hi=hi, center=center, scale=scale))
        elif spec.kind in ("bounded_0_100", "identity"):
            fitted.append(FittedTransform(spec=spec))
        else:
            raise ValueError(spec.kind)
    market_ft = None
    if market_train_raw is not None:
        spec   = FeatureTransformSpec(kind="robust_z", winsor_q=0.01, clip_value=8.0)
        v      = market_train_raw[:, 0]
        v      = v[np.isfinite(v)]
        lr     = np.log(v[1:] / v[:-1]) if v.size > 2 else np.array([0.0])
        lr     = lr[np.isfinite(lr)]
        lo, hi = _winsorize_fit(lr, spec.winsor_q)
        lr_w   = _winsorize_apply(lr, lo, hi)
        center = float(np.nanmedian(lr_w))
        mad    = float(np.nanmedian(np.abs(lr_w - center)))
        scale  = 1.4826 * mad
        if not np.isfinite(scale) or scale < 1e-6:
            scale = float(np.nanstd(lr_w))
        if not np.isfinite(scale) or scale < 1e-6:
            scale = 1.0
        market_ft = FittedTransform(spec=spec, lo=lo, hi=hi, center=center, scale=scale)
    return fitted, market_ft


def apply_transforms(
    X_raw: np.ndarray,
    fitted: List[FittedTransform],
    market_raw: Optional[np.ndarray],
    market_ft: Optional[FittedTransform],
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    X = np.empty_like(X_raw, dtype=np.float32)
    for j, ft in enumerate(fitted):
        X[:, :, j] = ft.transform(X_raw[:, :, j])
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    m_out = None
    if market_raw is not None and market_ft is not None:
        v      = market_raw[:, 0].astype(np.float32)
        lr     = np.full_like(v, np.nan)
        valid  = np.isfinite(v[1:]) & np.isfinite(v[:-1]) & (v[1:] > 0) & (v[:-1] > 0)
        lr[1:][valid] = np.log(v[1:][valid] / v[:-1][valid])
        lr     = np.nan_to_num(lr, nan=0.0, posinf=0.0, neginf=0.0)
        m_out  = market_ft.transform(lr).reshape(-1, 1).astype(np.float32)
        m_out  = np.nan_to_num(m_out, nan=0.0, posinf=0.0, neginf=0.0)
    return X, m_out

## 8) NaN strategy helpers

In [ ]:
def apply_nan_strategy(
    X_sentiment: np.ndarray,    # [T, n_sent_feats]
    nan_flag: np.ndarray,       # [T]  1 = data missing
    strategy: str,              # 'zero_fill' | 'carry_forward' | 'mask_asset'
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generic NaN strategy — works for both reddit and news flags.
    Returns (cleaned_X [T, n_sent_feats], loss_mask [T] bool).
    """
    missing = nan_flag.astype(bool)
    X_out   = X_sentiment.copy().astype(np.float32)
    loss_nan_mask = np.zeros(len(nan_flag), dtype=bool)

    if strategy == "zero_fill":
        X_out[missing] = 0.0
    elif strategy == "carry_forward":
        X_out[missing] = np.nan
        df_tmp = pd.DataFrame(X_out)
        X_out  = df_tmp.ffill().fillna(0.0).to_numpy(dtype=np.float32)
    elif strategy == "mask_asset":
        X_out[missing] = 0.0
        loss_nan_mask  = missing
    else:
        raise ValueError(f"Unknown nan_strategy: {strategy!r}")

    return X_out, loss_nan_mask

## 9) Raw arrays — merged sentiment

Builds `X_raw` with layout:
```
[ technical_feats (F) | reddit_sentiment_cols (R) | news_sentiment_cols (S) | _REDDIT_NAN | _NEWS_NAN ]
```
The combined NaN loss-mask is the union of both sources' masks (if either feed is down, that day is masked).

In [ ]:
def apply_spread_to_target(raw_ret: np.ndarray, spread: float) -> np.ndarray:
    return np.sign(raw_ret) * np.maximum(np.abs(raw_ret) - spread, 0.0)


def build_raw_arrays_merged(
    df: pd.DataFrame,
    assets: List[str],
    picks: Dict[str, Any],
    features: Dict[str, List[str]],
    logger: logging.Logger,
    reddit_sentiment_cols: List[str],   # e.g. ['reddit_threshold0.50_balance_ratio']
    news_sentiment_cols: List[str],     # e.g. ['news_threshold0.75_balance']
    reddit_nan_strategy: str,
    news_nan_strategy: str,
    spreads: Optional[np.ndarray] = None,
):
    """
    Build feature array for the merged model.

    Feature layout per asset  [T, N, F_total]:
      indices  0 .. F-1          : technical features
      indices  F .. F+R-1        : reddit sentiment (R cols)
      indices  F+R .. F+R+S-1    : news   sentiment (S cols)
      index    F+R+S             : _REDDIT_NAN flag
      index    F+R+S+1           : _NEWS_NAN   flag
      (market feature lives in a separate [T,1] array, prepended by SequenceDataset)

    Returns:
      closes_raw : [T, N]
      X_raw      : [T, N, F+R+S+2]
      y_model    : [T, N]  spread-adjusted target
      y_raw      : [T, N]  raw close-to-close return
      y_mask     : [T, N]  valid target mask
      feat_names : List[str]
      market_raw : [T, 1] or None
    """
    per_asset_feats = get_per_asset_feature_list(picks)

    cols = required_df_columns_merged(
        assets, per_asset_feats, picks,
        reddit_sentiment_cols, news_sentiment_cols,
    )
    missing = [c for c in cols if c not in df.columns]
    if missing:
        logger.error(f"Missing columns ({len(missing)}): {missing[:25]}"
                     f"{'...' if len(missing) > 25 else ''}")
        raise KeyError("Dataset missing required columns for merged feature set.")

    if spreads is None:
        spreads = np.zeros(len(assets), dtype=np.float32)
    else:
        spreads = np.asarray(spreads, dtype=np.float32)
        if spreads.shape[0] != len(assets):
            raise ValueError(f"spreads length {len(spreads)} != {len(assets)}")

    T = len(df)
    N = len(assets)
    F = len(per_asset_feats)
    R = len(reddit_sentiment_cols)
    S = len(news_sentiment_cols)
    # Total dims: technical + reddit_sent + news_sent + _REDDIT_NAN + _NEWS_NAN
    F_total = F + R + S + 2

    closes = np.zeros((T, N), dtype=np.float32)
    X_raw  = np.zeros((T, N, F_total), dtype=np.float32)

    # ── Load global NaN flags ─────────────────────────────────────────────
    reddit_nan_flag = (df["_REDDIT_NAN"].to_numpy(np.float32)
                       if reddit_sentiment_cols else np.zeros(T, np.float32))
    news_nan_flag   = (df["_NEWS_NAN"].to_numpy(np.float32)
                       if news_sentiment_cols   else np.zeros(T, np.float32))

    # Combined loss mask = union of both feeds being down
    _, reddit_loss_mask = apply_nan_strategy(
        np.zeros((T, max(R, 1)), dtype=np.float32), reddit_nan_flag, reddit_nan_strategy)
    _, news_loss_mask   = apply_nan_strategy(
        np.zeros((T, max(S, 1)), dtype=np.float32), news_nan_flag,   news_nan_strategy)
    combined_loss_mask = reddit_loss_mask | news_loss_mask  # [T] bool

    for i, a in enumerate(assets):
        c = df[f"{a}_Close"].to_numpy(np.float32)
        closes[:, i] = c

        # ── Technical features ────────────────────────────────────────────
        for j, f in enumerate(per_asset_feats):
            if f == "log_ret_1d":
                lr    = np.full_like(c, np.nan)
                valid = (c[1:] > 0) & (c[:-1] > 0) & np.isfinite(c[1:]) & np.isfinite(c[:-1])
                lr[1:][valid] = np.log(c[1:][valid] / c[:-1][valid])
                X_raw[:, i, j] = lr
            elif f in ("Open", "High", "Low"):
                raw   = df[f"{a}_{f}"].to_numpy(np.float32)
                rel   = np.full_like(raw, np.nan)
                valid = (raw > 0) & (c > 0) & np.isfinite(raw) & np.isfinite(c)
                rel[valid] = np.log(raw[valid] / c[valid])
                X_raw[:, i, j] = rel
            elif f.startswith("SMA"):
                sma   = df[f"{a}_{f}"].to_numpy(np.float32)
                rel   = np.full_like(sma, np.nan)
                valid = (sma > 0) & (c > 0) & np.isfinite(sma) & np.isfinite(c)
                rel[valid] = np.log(c[valid] / sma[valid])
                X_raw[:, i, j] = rel
            else:
                X_raw[:, i, j] = df[f"{a}_{f}"].to_numpy(np.float32)

        # ── Reddit sentiment ──────────────────────────────────────────────
        if R > 0:
            r_raw = np.stack(
                [df[f"{a}_{sf}"].to_numpy(np.float32) for sf in reddit_sentiment_cols],
                axis=1,
            )  # [T, R]
            r_clean, _ = apply_nan_strategy(r_raw, reddit_nan_flag, reddit_nan_strategy)
            X_raw[:, i, F:F + R] = r_clean

        # ── News sentiment ────────────────────────────────────────────────
        if S > 0:
            n_raw = np.stack(
                [df[f"{a}_{sf}"].to_numpy(np.float32) for sf in news_sentiment_cols],
                axis=1,
            )  # [T, S]
            n_clean, _ = apply_nan_strategy(n_raw, news_nan_flag, news_nan_strategy)
            X_raw[:, i, F + R:F + R + S] = n_clean

        # ── NaN flags (broadcast global to every asset) ───────────────────
        X_raw[:, i, F + R + S]     = reddit_nan_flag
        X_raw[:, i, F + R + S + 1] = news_nan_flag

    # ── Targets ───────────────────────────────────────────────────────────
    y_model = np.full((T, N), np.nan, dtype=np.float32)
    y_raw   = np.full((T, N), np.nan, dtype=np.float32)
    valid_y = np.zeros((T, N), dtype=bool)

    for i in range(N):
        c      = closes[:, i]
        valid  = (np.isfinite(c[:-1]) & np.isfinite(c[1:]) &
                  (c[:-1] > 0.0) & (c[1:] > 0.0))
        tmp_raw   = np.full(T - 1, np.nan, dtype=np.float32)
        tmp_model = np.full(T - 1, np.nan, dtype=np.float32)
        raw_ret   = (c[1:][valid] / c[:-1][valid]) - 1.0
        model_ret = apply_spread_to_target(raw_ret, float(spreads[i]))
        tmp_raw[valid]   = raw_ret.astype(np.float32)
        tmp_model[valid] = model_ret.astype(np.float32)
        y_raw[:-1, i]    = tmp_raw
        y_model[:-1, i]  = tmp_model
        valid_y[:-1, i]  = valid

    y_raw   = np.where(np.isfinite(y_raw),   y_raw,   np.nan).astype(np.float32)
    y_model = np.where(np.isfinite(y_model), y_model, np.nan).astype(np.float32)

    # Narrow y_mask by the combined NaN loss mask
    y_mask = valid_y & ~combined_loss_mask[:, None]

    market = None
    if picks.get("Market") is not None:
        market = df[picks["Market"]].to_numpy(np.float32).reshape(-1, 1)

    # Full feature name list (for transform dispatch)
    feat_names = (
        list(per_asset_feats)
        + list(reddit_sentiment_cols)
        + list(news_sentiment_cols)
        + ["_REDDIT_NAN", "_NEWS_NAN"]
    )

    return closes, X_raw, y_model, y_raw, y_mask, feat_names, market


def compute_sample_validity(
    X_raw: np.ndarray,
    y_mask: np.ndarray,
    lookback: int,
    min_valid_targets: int = 1,
) -> np.ndarray:
    T, N, F  = X_raw.shape
    finite_feats = np.isfinite(X_raw).all(axis=2)
    sample_ok    = np.zeros(T, dtype=bool)
    for t in range(lookback - 1, T - 1):
        win_ok    = finite_feats[t - lookback + 1: t + 1].all()
        target_ok = int(y_mask[t].sum()) >= min_valid_targets
        sample_ok[t] = bool(win_ok and target_ok)
    return sample_ok

## 10) Dataset & DataLoader  *(unchanged)*

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y, y_mask, valid_t, lookback, market=None):
        self.X       = X
        self.y       = y
        self.y_mask  = y_mask
        self.valid_t = np.asarray(valid_t, dtype=np.int64)
        self.lookback= int(lookback)
        self.market  = market

    def __len__(self):
        return len(self.valid_t)

    def __getitem__(self, idx):
        t = int(self.valid_t[idx])
        x = self.X[t - self.lookback + 1: t + 1]
        if self.market is not None:
            m = self.market[t - self.lookback + 1: t + 1]
            m = np.repeat(m[:, None, :], x.shape[1], axis=1)
            x = np.concatenate([x, m], axis=2)
        y = self.y[t].copy()
        m = self.y_mask[t].copy()
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        m = np.nan_to_num(m, nan=0.0, posinf=0.0, neginf=0.0)
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(m, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )


def make_loader(X, y, y_mask, valid_t, lookback, batch_size, market, shuffle):
    ds = SequenceDataset(X=X, y=y, y_mask=y_mask, valid_t=valid_t,
                         lookback=lookback, market=market)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## 11) Model — JointAssetLSTM  *(unchanged)*

In [ ]:
class CrossAssetMixer(nn.Module):
    def __init__(self, d_model, num_heads, dropout, num_layers):
        super().__init__()
        if num_layers <= 0:
            self.net = nn.Identity()
        else:
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=num_heads,
                dim_feedforward=max(4 * d_model, 64),
                dropout=dropout, batch_first=True,
                activation="gelu", norm_first=True,
            )
            self.net = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x):
        return self.net(x)


class JointAssetLSTM(nn.Module):
    def __init__(self, input_dim_per_asset, num_assets, hidden_size, num_layers,
                 dropout, bidirectional=False, encoder_proj=0,
                 asset_mixer_layers=1, asset_mixer_heads=4,
                 fusion_hidden=64, use_residual_fusion=True):
        super().__init__()
        self.num_assets = num_assets
        lstm_dropout    = dropout if num_layers > 1 else 0.0
        self.encoder    = nn.LSTM(
            input_size=input_dim_per_asset, hidden_size=hidden_size,
            num_layers=num_layers, dropout=lstm_dropout,
            batch_first=True, bidirectional=bidirectional,
        )
        enc_dim = hidden_size * (2 if bidirectional else 1)
        if encoder_proj > 0:
            self.asset_proj = nn.Sequential(
                nn.Linear(enc_dim, encoder_proj), nn.GELU(), nn.Dropout(dropout))
            asset_dim = encoder_proj
        else:
            self.asset_proj = nn.Identity()
            asset_dim       = enc_dim
        if asset_mixer_layers > 0:
            valid_heads = [h for h in [1, 2, 4, 8] if asset_dim % h == 0]
            if not valid_heads:                   asset_mixer_heads = 1
            elif asset_mixer_heads not in valid_heads: asset_mixer_heads = max(valid_heads)
        self.asset_mixer = CrossAssetMixer(
            d_model=asset_dim,
            num_heads=asset_mixer_heads if asset_mixer_layers > 0 else 1,
            dropout=dropout, num_layers=asset_mixer_layers,
        )
        self.use_residual_fusion = use_residual_fusion
        fusion_in = num_assets * asset_dim
        if fusion_hidden > 0:
            self.head = nn.Sequential(
                nn.Linear(fusion_in, fusion_hidden), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(fusion_hidden, num_assets))
        else:
            self.head = nn.Linear(fusion_in, num_assets)
        if use_residual_fusion:
            self.residual_head = nn.Linear(asset_dim, 1)

    def forward(self, x):
        B, L, N, F = x.shape
        x      = x.permute(0, 2, 1, 3).contiguous().reshape(B * N, L, F)
        out, _ = self.encoder(x)
        h      = out[:, -1, :]
        h      = self.asset_proj(h)
        h      = h.reshape(B, N, -1)
        z      = self.asset_mixer(h)
        yhat   = self.head(z.reshape(B, -1))
        if self.use_residual_fusion:
            res  = self.residual_head(h).squeeze(-1)
            yhat = yhat + res
        return yhat

## 12) Loss & metrics  *(unchanged)*

In [ ]:
def masked_mse_loss(pred, target, mask):
    mask        = mask.float()
    target_safe = torch.where(mask > 0, target, torch.zeros_like(target))
    pred_safe   = torch.where(mask > 0, pred,   torch.zeros_like(pred))
    err2        = (pred_safe - target_safe) ** 2 * mask
    loss        = err2.sum() / torch.clamp(mask.sum(), min=1.0)
    if not torch.isfinite(loss):
        raise ValueError("masked_mse_loss became non-finite.")
    return loss


def masked_huber_loss(pred, target, mask, delta=0.01):
    mask        = mask.float()
    target_safe = torch.where(mask > 0, target, torch.zeros_like(target))
    pred_safe   = torch.where(mask > 0, pred,   torch.zeros_like(pred))
    err         = pred_safe - target_safe
    abs_err     = torch.abs(err)
    delta_t     = torch.tensor(delta, dtype=pred.dtype, device=pred.device)
    quad        = torch.minimum(abs_err, delta_t)
    loss        = (0.5 * quad ** 2 + delta_t * (abs_err - quad)) * mask
    loss        = loss.sum() / torch.clamp(mask.sum(), min=1.0)
    if not torch.isfinite(loss):
        raise ValueError("masked_huber_loss became non-finite.")
    return loss


def regression_metrics(y_true, y_pred, mask):
    valid = mask.astype(bool)
    yt, yp = y_true[valid], y_pred[valid]
    if yt.size == 0:
        return {k: np.nan for k in
                ["rmse", "mae", "r2", "directional_acc", "ic_pearson", "ic_spearman"]}
    rmse   = float(np.sqrt(np.mean((yp - yt) ** 2)))
    mae    = float(np.mean(np.abs(yp - yt)))
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - np.mean(yt)) ** 2))
    r2     = float(1.0 - ss_res / ss_tot) if ss_tot > 1e-12 else np.nan
    dir_acc= float(np.mean(np.sign(yp) == np.sign(yt)))
    if yt.size > 1 and np.std(yt) > 1e-12 and np.std(yp) > 1e-12:
        ic_p   = float(np.corrcoef(yt, yp)[0, 1])
        ry, rp = pd.Series(yt).rank().to_numpy(), pd.Series(yp).rank().to_numpy()
        ic_s   = float(np.corrcoef(ry, rp)[0, 1]) if np.std(ry)>1e-12 and np.std(rp)>1e-12 else np.nan
    else:
        ic_p = ic_s = np.nan
    return {"rmse": rmse, "mae": mae, "r2": r2,
            "directional_acc": dir_acc, "ic_pearson": ic_p, "ic_spearman": ic_s}


def per_asset_metrics(y_true, y_pred, mask, asset_names):
    return [{"asset": a, **regression_metrics(y_true[:, [i]], y_pred[:, [i]], mask[:, [i]])}
            for i, a in enumerate(asset_names)]

## 13) Training loop  *(unchanged)*

In [ ]:
def evaluate_model(model, loader, device, loss_name="mse", huber_delta=0.01):
    model.eval()
    ys, yhs, ms, ts, losses = [], [], [], [], []
    with torch.no_grad():
        for xb, yb, mb, tb in loader:
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
            pred        = model(xb)
            loss        = (masked_huber_loss(pred, yb, mb, delta=huber_delta)
                           if loss_name == "huber" else masked_mse_loss(pred, yb, mb))
            losses.append(float(loss.item()))
            ys.append(yb.cpu().numpy())
            yhs.append(pred.cpu().numpy())
            ms.append(mb.cpu().numpy())
            ts.append(tb.cpu().numpy())
    y_true  = np.concatenate(ys,  axis=0) if ys  else np.empty((0, 0), dtype=np.float32)
    y_pred  = np.concatenate(yhs, axis=0) if yhs else np.empty((0, 0), dtype=np.float32)
    mask    = np.concatenate(ms,  axis=0) if ms  else np.empty((0, 0), dtype=np.float32)
    t_idx   = np.concatenate(ts,  axis=0) if ts  else np.empty((0,),   dtype=np.int64)
    metrics = regression_metrics(y_true, y_pred, mask)
    metrics["loss"] = float(np.mean(losses)) if losses else np.nan
    return {"metrics": metrics, "y_true": y_true, "y_pred": y_pred,
            "mask": mask, "t_idx": t_idx}


def train_one_model(X, y, y_mask, market, train_t, es_t, lookback,
                    input_dim_per_asset, output_dim, params, seed, logger):
    seed_everything(seed)
    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader = make_loader(X, y, y_mask, train_t, lookback,
                               params["batch_size"], market, shuffle=True)
    es_loader    = make_loader(X, y, y_mask, es_t, lookback,
                               params["batch_size"], market, shuffle=False)
    model = JointAssetLSTM(
        input_dim_per_asset=input_dim_per_asset, num_assets=output_dim,
        hidden_size=params["hidden_size"], num_layers=params["num_layers"],
        dropout=params["dropout"], bidirectional=params["bidirectional"],
        encoder_proj=params["encoder_proj"], asset_mixer_layers=params["asset_mixer_layers"],
        asset_mixer_heads=params["asset_mixer_heads"], fusion_hidden=params["fusion_hidden"],
        use_residual_fusion=params["use_residual_fusion"],
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params["learning_rate"],
        weight_decay=params["weight_decay"])
    if params["scheduler"] == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(params["max_epochs"], 5))
    elif params["scheduler"] == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3)
    else:
        scheduler = None

    best_state, best_score, best_epoch, patience_count = None, np.inf, -1, 0
    for epoch in range(1, params["max_epochs"] + 1):
        model.train()
        batch_losses = []
        for xb, yb, mb, _ in train_loader:
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
            if not torch.isfinite(xb).all(): raise ValueError("Non-finite xb.")
            if mb.sum() <= 0: continue
            if not torch.isfinite(yb[mb > 0]).all(): raise ValueError("Non-finite targets.")
            optimizer.zero_grad()
            pred = model(xb)
            if not torch.isfinite(pred).all(): raise ValueError("Non-finite predictions.")
            loss = (masked_huber_loss(pred, yb, mb, delta=params["huber_delta"])
                    if params["loss_name"] == "huber" else masked_mse_loss(pred, yb, mb))
            if not torch.isfinite(loss): raise ValueError("Non-finite loss.")
            loss.backward()
            if params["grad_clip"] and params["grad_clip"] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), params["grad_clip"])
            optimizer.step()
            batch_losses.append(float(loss.item()))
        train_loss = float(np.mean(batch_losses)) if batch_losses else np.nan
        es_eval    = evaluate_model(model, es_loader, device,
                                    params["loss_name"], params["huber_delta"])
        es_loss    = float(es_eval["metrics"]["loss"])
        logger.info(f"[TRAIN] seed={seed} epoch={epoch:03d} "
                    f"train_loss={train_loss:.6f} es_loss={es_loss:.6f}")
        if scheduler is not None:
            scheduler.step(es_loss) if params["scheduler"] == "plateau" else scheduler.step()
        if np.isfinite(es_loss) and es_loss < (best_score - params["min_delta"]):
            best_score, best_epoch, best_state, patience_count = (
                es_loss, epoch, copy.deepcopy(model.state_dict()), 0)
        else:
            patience_count += 1
        if epoch >= params["min_epochs"] and patience_count >= params["patience"]:
            logger.info(f"[EARLY STOP] seed={seed} best_epoch={best_epoch}")
            break
    if best_state is None:
        raise ValueError("No valid best_state found.")
    model.load_state_dict(best_state)
    return model, {}


def evaluate_model_on_t_indices(
    model, X, y, y_mask, market, t_idx, lookback, batch_size, loss_name, huber_delta
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loader = make_loader(X, y, y_mask, t_idx, lookback, batch_size, market, shuffle=False)
    return evaluate_model(model, loader, device, loss_name, huber_delta)


def nanmean_predictions(pred_list):
    return np.nanmean(np.stack(pred_list, axis=0), axis=0).astype(np.float32)


def train_and_eval_one_split_seed(
    X_transformed, y_model, y_mask, market_transformed,
    asset_names, dates_index, train_idx, val_idx,
    lookback, seed, model_params, trial_dir, logger, split_i, save_artifacts=False,
):
    train_core_idx, es_idx = split_train_for_early_stop(
        train_idx, dates_index, es_years=1, min_train_days=252)
    sample_valid = compute_sample_validity(
        X_transformed, y_mask, lookback,
        min_valid_targets=max(1, len(asset_names) // 3))
    train_t = train_core_idx[sample_valid[train_core_idx]]
    es_t    = es_idx[sample_valid[es_idx]]
    val_t   = val_idx[sample_valid[val_idx]]
    if len(train_t) < 64 or len(es_t) < 32 or len(val_t) < 32:
        raise optuna.exceptions.TrialPruned()
    if not np.isfinite(X_transformed).all():
        raise ValueError("X_transformed contains non-finite values.")
    if market_transformed is not None and not np.isfinite(market_transformed).all():
        raise ValueError("market_transformed contains non-finite values.")
    input_dim_per_asset = X_transformed.shape[2] + (1 if market_transformed is not None else 0)
    output_dim = len(asset_names)
    logger.info(f"[SPLIT {split_i}] seed={seed} "
                f"train={len(train_t)} es={len(es_t)} val={len(val_t)} "
                f"input_dim={input_dim_per_asset}")
    model, _ = train_one_model(
        X=X_transformed, y=y_model, y_mask=y_mask, market=market_transformed,
        train_t=train_t, es_t=es_t, lookback=lookback,
        input_dim_per_asset=input_dim_per_asset, output_dim=output_dim,
        params=model_params, seed=seed, logger=logger,
    )
    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    val_loader = make_loader(X_transformed, y_model, y_mask, val_t, lookback,
                             model_params["batch_size"], market_transformed, shuffle=False)
    val_eval   = evaluate_model(model, val_loader, device,
                                model_params["loss_name"], model_params["huber_delta"])
    return val_eval

## 14) Per-rank merged Optuna objective

**What's frozen**: all feature selections (technical + both sentiment sources, their thresholds,
NaN strategies) — these come directly from the reddit and news best trials.

**What Optuna tunes**: all LSTM architecture + training hyperparameters.

In [ ]:
def _build_model_params_from_trial(t: optuna.trial.FrozenTrial) -> Dict[str, Any]:
    p = t.params
    return {
        "hidden_size":         p["hidden_size"],
        "num_layers":          p["num_layers"],
        "dropout":             p["dropout"],
        "bidirectional":       bool(p["bidirectional"]),
        "encoder_proj":        p["encoder_proj"],
        "asset_mixer_layers":  p["asset_mixer_layers"],
        "asset_mixer_heads":   p["asset_mixer_heads"],
        "fusion_hidden":       p["fusion_hidden"],
        "use_residual_fusion": bool(p["use_residual_fusion"]),
        "scheduler":           p["scheduler"],
        "learning_rate":       p["learning_rate"],
        "weight_decay":        p["weight_decay"],
        "batch_size":          p["batch_size"],
        "max_epochs":          p["max_epochs"],
        "patience":            p["patience"],
        "min_epochs":          p["min_epochs"],
        "grad_clip":           p["grad_clip"],
        "loss_name":           p["loss_name"],
        "huber_delta":         p["huber_delta"],
        "min_delta":           p["min_delta"],
    }


def per_lstm_objective_factory(
    df: pd.DataFrame,
    assets: List[str],
    merged_config: Dict[str, Any],   # single merged config for this rank
    base_log_dir: str,
    seeds: List[int],
    spreads: Optional[np.ndarray] = None,
):
    """
    Objective for ONE merged LSTM rank.
    All feature picks are fully frozen from merged_config.
    Optuna searches only LSTM architecture + training hyperparameters.
    """
    dates_index = pd.DatetimeIndex(df.index)
    splits      = make_walk_forward_splits(dates_index)

    picks                 = merged_config["tech_picks"]
    reddit_sentiment_cols = merged_config["reddit_sentiment_cols"]
    news_sentiment_cols   = merged_config["news_sentiment_cols"]
    reddit_nan_strategy   = merged_config["reddit_nan_strategy"]
    news_nan_strategy     = merged_config["news_nan_strategy"]

    def objective(trial: optuna.Trial) -> float:
        trial_dir = os.path.join(base_log_dir, f"trial_{trial.number:05d}")
        logger    = setup_logger(trial_dir)
        logger.info("=" * 100)
        logger.info(
            f"TRIAL {trial.number} | rank={merged_config['rank']}"
            f" | reddit_trial={merged_config['reddit_trial_number']}"
            f" IC={merged_config['reddit_ic']:.5f}"
            f" | news_trial={merged_config['news_trial_number']}"
            f" IC={merged_config['news_ic']:.5f}"
        )
        logger.info(f"Frozen reddit_sentiment: {reddit_sentiment_cols}")
        logger.info(f"Frozen news_sentiment:   {news_sentiment_cols}")

        # ── Store frozen config as user attrs for downstream use ──────────
        trial.set_user_attr("merged_config",       merged_config)
        trial.set_user_attr("tech_picks",          picks)
        trial.set_user_attr("reddit_sentiment_cols", reddit_sentiment_cols)
        trial.set_user_attr("news_sentiment_cols",   news_sentiment_cols)
        trial.set_user_attr("reddit_nan_strategy", reddit_nan_strategy)
        trial.set_user_attr("news_nan_strategy",   news_nan_strategy)

        # ── LSTM + training HPs (the only search space) ───────────────────
        lookback    = trial.suggest_categorical("lookback",    [5, 10, 22])
        hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
        num_layers  = trial.suggest_int("num_layers", 1, 2)
        dropout     = trial.suggest_float("dropout", 0.0, 0.3)
        bidirectional      = bool(trial.suggest_categorical("bidirectional",      [0, 1]))
        encoder_proj       = trial.suggest_categorical("encoder_proj",       [0, 32, 64])
        asset_mixer_layers = trial.suggest_categorical("asset_mixer_layers", [0, 1, 2])
        asset_mixer_heads  = trial.suggest_categorical("asset_mixer_heads",  [1, 2, 4])
        fusion_hidden      = trial.suggest_categorical("fusion_hidden",      [0, 64, 128])
        use_residual_fusion= bool(trial.suggest_categorical("use_residual_fusion", [0, 1]))
        scheduler    = trial.suggest_categorical("scheduler",    ["none", "plateau"])
        learning_rate= trial.suggest_float("learning_rate", 1e-4, 2e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay",  1e-6, 1e-3, log=True)
        batch_size   = trial.suggest_categorical("batch_size",   [32, 64, 128])
        max_epochs   = trial.suggest_categorical("max_epochs",   [20, 30, 40])
        patience     = trial.suggest_categorical("patience",     [5, 7])
        min_epochs   = trial.suggest_categorical("min_epochs",   [5, 8])
        grad_clip    = trial.suggest_categorical("grad_clip",    [0.5, 1.0, 2.0])
        loss_name    = trial.suggest_categorical("loss_name",    ["mse", "huber"])
        huber_delta  = trial.suggest_categorical("huber_delta",  [0.005, 0.01, 0.02])
        min_delta    = trial.suggest_categorical("min_delta",    [0.0, 1e-5])

        model_params = {
            "hidden_size": hidden_size, "num_layers": num_layers, "dropout": dropout,
            "bidirectional": bidirectional, "encoder_proj": encoder_proj,
            "asset_mixer_layers": asset_mixer_layers, "asset_mixer_heads": asset_mixer_heads,
            "fusion_hidden": fusion_hidden, "use_residual_fusion": use_residual_fusion,
            "scheduler": scheduler, "learning_rate": learning_rate,
            "weight_decay": weight_decay, "batch_size": batch_size,
            "max_epochs": max_epochs, "patience": patience, "min_epochs": min_epochs,
            "grad_clip": grad_clip, "loss_name": loss_name,
            "huber_delta": huber_delta, "min_delta": min_delta,
        }

        # ── Build merged feature arrays ───────────────────────────────────
        try:
            _, X_raw, y_model, _, y_mask, feat_names, market_raw =                 build_raw_arrays_merged(
                    df=df, assets=assets, picks=picks, features=FEATURES,
                    logger=logger,
                    reddit_sentiment_cols=reddit_sentiment_cols,
                    news_sentiment_cols=news_sentiment_cols,
                    reddit_nan_strategy=reddit_nan_strategy,
                    news_nan_strategy=news_nan_strategy,
                    spreads=spreads,
                )
        except KeyError as e:
            logger.info(f"[PRUNE] Missing columns: {e}")
            raise optuna.TrialPruned()

        all_rmse, all_ic, all_scores = [], [], []

        for split_i, (train_idx, val_idx) in enumerate(splits):
            logger.info("-" * 80)
            logger.info(f"[SPLIT {split_i}] "
                        f"train {df.index[train_idx[0]]}..{df.index[train_idx[-1]]} | "
                        f"val   {df.index[val_idx[0]]}..{df.index[val_idx[-1]]}")

            X_train_raw      = X_raw[train_idx]
            market_train_raw = market_raw[train_idx] if market_raw is not None else None
            fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
            X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

            split_seed_scores = []
            for seed in seeds:
                try:
                    res = train_and_eval_one_split_seed(
                        X_transformed=X_trans, y_model=y_model, y_mask=y_mask,
                        market_transformed=market_trans, asset_names=assets,
                        dates_index=dates_index, train_idx=train_idx, val_idx=val_idx,
                        lookback=lookback, seed=seed, model_params=model_params,
                        trial_dir=trial_dir, logger=logger,
                        split_i=split_i, save_artifacts=False,
                    )
                except ValueError as e:
                    logger.info(f"[PRUNE] Non-finite: {e}")
                    raise optuna.TrialPruned()

                ic_s = float(res["metrics"]["ic_spearman"])                     if np.isfinite(res["metrics"]["ic_spearman"]) else -999.0
                rmse = float(res["metrics"]["rmse"])
                all_rmse.append(rmse); all_ic.append(ic_s)
                all_scores.append(ic_s); split_seed_scores.append(ic_s)
                logger.info(f"[SPLIT {split_i}][SEED {seed}] ic_s={ic_s:.6f} rmse={rmse:.6f}")

            split_mean = float(np.mean(split_seed_scores))
            trial.report(split_mean, step=split_i)
            if trial.should_prune():
                logger.info("[PRUNE] Pruned by MedianPruner.")
                raise optuna.TrialPruned()

        final_score = float(np.mean(all_scores))
        trial.set_user_attr("mean_ic",   float(np.mean(all_ic)))
        trial.set_user_attr("mean_rmse", float(np.mean(all_rmse)))
        trial.set_user_attr("all_scores", [float(x) for x in all_scores])
        logger.info(f"TRIAL {trial.number} DONE | score={final_score:.6f}")
        return final_score

    return objective


def run_per_lstm_search(
    df: pd.DataFrame,
    assets: List[str],
    merged_configs: List[Dict[str, Any]],
    out_dir: str,
    n_trials: int,
    seeds: List[int],
    spreads: Optional[np.ndarray] = None,
) -> List[optuna.Study]:
    """
    Run one independent Optuna study per rank.
    DB saved to: {out_dir}/lstm_top{rank}/lstm_top{rank}.db
    Returns list of studies in rank order.
    """
    studies = []
    for rank, merged_config in enumerate(merged_configs):
        rank_dir   = os.path.join(out_dir, f"lstm_top{rank}")
        safe_makedirs(rank_dir)
        db_path    = os.path.join(rank_dir, f"lstm_top{rank}.db")
        storage    = f"sqlite:///{db_path}"
        study_name = f"lstm_top{rank}"

        print(f"\n{'='*70}")
        print(f"Rank {rank}"
              f" | reddit IC={merged_config['reddit_ic']:.5f}"
              f" | news IC={merged_config['news_ic']:.5f}")
        print(f"  reddit_sent: {merged_config['reddit_sentiment_cols']}")
        print(f"  news_sent:   {merged_config['news_sentiment_cols']}")
        print(f"  DB: {db_path}")

        study = optuna.create_study(
            study_name=study_name,
            direction="maximize",
            storage=storage,
            load_if_exists=True,
            sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        )
        objective = per_lstm_objective_factory(
            df=df, assets=assets, merged_config=merged_config,
            base_log_dir=rank_dir, seeds=seeds, spreads=spreads,
        )
        study.optimize(objective, n_trials=n_trials, gc_after_trial=True,
                       show_progress_bar=True)

        best = study.best_trial
        with open(os.path.join(rank_dir, "best_trial.json"), "w") as fh:
            json.dump({
                "rank":             rank,
                "best_value":       best.value,
                "best_params":      best.params,
                "best_user_attrs":  {k: v for k, v in best.user_attrs.items()
                                     if isinstance(v, (str, int, float, list, dict, bool))},
                "n_trials":         len(study.trials),
            }, fh, indent=2)

        print(f"  Best merged IC: {best.value:.5f}")
        studies.append(study)

    return studies

## 15) Black–Litterman  *(unchanged)*

In [ ]:
def _safe_float(x):
    try:
        v = float(x)
        return v if np.isfinite(v) else None
    except Exception:
        return None


def estimate_train_covariance(y_train, y_mask_train, ridge=1e-5):
    y_df   = pd.DataFrame(np.where(y_mask_train, y_train, np.nan))
    Sigma  = y_df.cov(min_periods=max(20, len(y_df) // 10)).to_numpy(dtype=np.float64)
    Sigma  = np.nan_to_num(Sigma, nan=0.0, posinf=0.0, neginf=0.0)
    diag   = np.diag(Sigma).copy()
    pos    = diag[np.isfinite(diag) & (diag > 0)]
    fill   = float(np.median(pos)) if len(pos) > 0 else 1e-4
    for i in range(len(diag)):
        if not np.isfinite(Sigma[i, i]) or Sigma[i, i] <= 0:
            Sigma[i, i] = fill
    Sigma = 0.5 * (Sigma + Sigma.T) + ridge * np.eye(Sigma.shape[0])
    return Sigma


def black_litterman_posterior(
    Sigma, q, market_weights=None, tau=0.05, risk_aversion=2.5,
    omega_scale=0.25, ridge=1e-8,
):
    N       = Sigma.shape[0]
    Sigma   = 0.5 * (Sigma + Sigma.T) + ridge * np.eye(N)
    w_mkt   = (np.full(N, 1.0 / N) if market_weights is None
               else np.clip(market_weights, 0, None))
    if w_mkt.sum() <= 0: w_mkt = np.full(N, 1.0 / N)
    else: w_mkt = w_mkt / w_mkt.sum()
    q       = np.asarray(q, dtype=np.float64)
    pi      = risk_aversion * (Sigma @ w_mkt)
    P       = np.eye(N)
    tauSig  = tau * Sigma
    Omega   = np.diag(np.maximum(np.diag(tauSig) * omega_scale, ridge))
    A       = np.linalg.inv(tauSig) + P.T @ np.linalg.inv(Omega) @ P
    b       = np.linalg.inv(tauSig) @ pi + P.T @ np.linalg.inv(Omega) @ q
    return np.linalg.solve(A, b), pi, Omega


def mean_variance_weights(
    mu, Sigma, risk_aversion=2.5, long_only=True, max_weight=None, ridge=1e-8
):
    A = risk_aversion * Sigma + ridge * np.eye(len(mu))
    try: w = np.linalg.solve(A, mu)
    except np.linalg.LinAlgError: w = np.full(len(mu), 1.0 / len(mu))
    if long_only: w = np.clip(w, 0, None)
    if max_weight and max_weight > 0: w = np.minimum(w, max_weight)
    return w / w.sum() if w.sum() > 0 else np.full(len(mu), 1.0 / len(mu))


def portfolio_metrics_from_returns(rets, freq=252):
    rets = np.asarray(rets)[np.isfinite(rets)]
    if rets.size == 0:
        return {k: np.nan for k in ["n_days","mean_daily_return","vol_daily",
                                    "ann_return","ann_vol","sharpe","cum_return","max_drawdown"]}
    ann_r = float(np.mean(rets) * freq)
    ann_v = float(np.std(rets) * np.sqrt(freq))
    cum   = float(np.prod(1 + rets) - 1)
    roll  = np.cumprod(1 + rets)
    dd    = float(np.min(roll / np.maximum.accumulate(roll)) - 1)
    return {"n_days": len(rets), "mean_daily_return": float(np.mean(rets)),
            "vol_daily": float(np.std(rets)), "ann_return": ann_r,
            "ann_vol": ann_v, "sharpe": ann_r / ann_v if ann_v > 1e-8 else np.nan,
            "cum_return": cum, "max_drawdown": dd}


def run_black_litterman_backtest(
    dates_target, y_true, y_pred, mask, asset_names,
    train_cov, tau=0.05, risk_aversion=2.5, omega_scale=0.25,
    long_only=True, max_weight=None,
):
    T, N = y_true.shape
    weights_full = np.zeros((T, N), dtype=np.float64)
    port_rets    = np.full(T, np.nan, dtype=np.float64)
    for t in range(T):
        active = mask[t].astype(bool) & np.isfinite(y_pred[t]) & np.isfinite(y_true[t])
        if active.sum() == 0: continue
        idx      = np.where(active)[0]
        q        = y_pred[t, idx].astype(np.float64)
        Sigma_sub= train_cov[np.ix_(idx, idx)]
        mu_bl, _, _ = black_litterman_posterior(
            Sigma=Sigma_sub, q=q, market_weights=None, tau=tau,
            risk_aversion=risk_aversion, omega_scale=omega_scale,
        )
        w_sub = mean_variance_weights(
            mu=mu_bl, Sigma=Sigma_sub, risk_aversion=risk_aversion,
            long_only=long_only, max_weight=max_weight,
        )
        w = np.zeros(N, dtype=np.float64)
        w[idx] = w_sub
        weights_full[t] = w
        port_rets[t] = float(np.dot(w, np.nan_to_num(y_true[t].astype(np.float64), nan=0.0)))
    weights_df = pd.DataFrame(weights_full, index=dates_target, columns=asset_names)
    returns_df = pd.DataFrame(
        {"portfolio_return": port_rets,
         "portfolio_wealth": np.cumprod(1.0 + np.nan_to_num(port_rets, nan=0.0))},
        index=dates_target,
    )
    if len(weights_df) > 1:
        avg_turnover = float(np.abs(weights_df.diff().fillna(0.0)).sum(axis=1).mean())
    else:
        avg_turnover = np.nan
    pm = portfolio_metrics_from_returns(port_rets[np.isfinite(port_rets)])
    pm["avg_daily_turnover"] = avg_turnover
    return {"metrics": pm, "weights_df": weights_df, "returns_df": returns_df}

## 16) BL ensemble search  *(unchanged pattern)*

In [ ]:
def _build_topn_ensemble_for_bl(
    df, assets, lstm_studies, split_train_idx, split_val_idx,
    dates_index, seed, logger, spreads=None,
):
    T = len(df)
    N = len(assets)
    pred_full_list   = []
    canonical_y_raw  = None
    canonical_y_mask = None

    for rank, study in enumerate(lstm_studies):
        best = study.best_trial
        picks                 = best.user_attrs["tech_picks"]
        reddit_sentiment_cols = best.user_attrs["reddit_sentiment_cols"]
        news_sentiment_cols   = best.user_attrs["news_sentiment_cols"]
        reddit_nan_strategy   = best.user_attrs["reddit_nan_strategy"]
        news_nan_strategy     = best.user_attrs["news_nan_strategy"]
        lookback              = int(best.params["lookback"])
        model_params          = _build_model_params_from_trial(best)

        _, X_raw, y_model, y_raw, y_mask, feat_names, market_raw =             build_raw_arrays_merged(
                df=df, assets=assets, picks=picks, features=FEATURES,
                logger=logger,
                reddit_sentiment_cols=reddit_sentiment_cols,
                news_sentiment_cols=news_sentiment_cols,
                reddit_nan_strategy=reddit_nan_strategy,
                news_nan_strategy=news_nan_strategy,
                spreads=spreads,
            )

        if canonical_y_raw is None:
            canonical_y_raw  = y_raw.copy()
            canonical_y_mask = y_mask.copy()

        X_train_raw       = X_raw[split_train_idx]
        market_train_raw  = market_raw[split_train_idx] if market_raw is not None else None
        fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
        X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

        sample_valid = compute_sample_validity(
            X_trans, y_mask, lookback, min_valid_targets=max(1, N // 3))
        train_core_idx, es_idx = split_train_for_early_stop(
            split_train_idx, dates_index, es_years=1, min_train_days=252)
        train_core_t = train_core_idx[sample_valid[train_core_idx]]
        es_t         = es_idx[sample_valid[es_idx]]
        val_t        = split_val_idx[sample_valid[split_val_idx]]

        if len(train_core_t) < 64 or len(es_t) < 32 or len(val_t) < 32:
            logger.warning(f"Rank {rank}: too few samples, skipping.")
            continue

        input_dim = X_trans.shape[2] + (1 if market_trans is not None else 0)
        model, _  = train_one_model(
            X=X_trans, y=y_model, y_mask=y_mask, market=market_trans,
            train_t=train_core_t, es_t=es_t, lookback=lookback,
            input_dim_per_asset=input_dim, output_dim=N,
            params=model_params, seed=seed, logger=logger,
        )
        val_eval = evaluate_model_on_t_indices(
            model=model, X=X_trans, y=y_model, y_mask=y_mask,
            market=market_trans, t_idx=val_t, lookback=lookback,
            batch_size=model_params["batch_size"],
            loss_name=model_params["loss_name"],
            huber_delta=model_params["huber_delta"],
        )
        pred_full = np.full((T, N), np.nan, dtype=np.float32)
        if len(val_eval["t_idx"]) > 0:
            pred_full[val_eval["t_idx"]] = val_eval["y_pred"]
        pred_full_list.append(pred_full)
        logger.info(f"Rank {rank}: val IC_s={val_eval['metrics']['ic_spearman']:.4f}")

    if not pred_full_list:
        raise ValueError("No usable predictions from any rank.")
    ensemble  = nanmean_predictions(pred_full_list)
    train_cov = estimate_train_covariance(
        canonical_y_raw[split_train_idx], canonical_y_mask[split_train_idx])
    return {"ensemble_pred": ensemble, "y_raw": canonical_y_raw,
            "y_mask": canonical_y_mask, "train_cov": train_cov}


def run_bl_search(
    df, assets, lstm_studies, out_dir, n_trials_bl, seed=SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_REDDIT_NEWS, spreads=None,
):
    safe_makedirs(out_dir)
    dates_index = pd.DatetimeIndex(df.index)
    splits      = make_walk_forward_splits(dates_index)
    db_path     = os.path.join(out_dir, "black_litterman_optuna.db")
    storage     = f"sqlite:///{db_path}"

    def bl_objective(trial: optuna.Trial) -> float:
        tau           = trial.suggest_float("tau",           1e-3, 0.5, log=True)
        risk_aversion = trial.suggest_float("risk_aversion", 0.5,  10.0, log=True)
        omega_scale   = trial.suggest_float("omega_scale",   0.01, 2.0,  log=True)
        max_weight    = trial.suggest_float("max_weight",    0.10, 1.0)
        long_only     = True
        logger = setup_logger(os.path.join(out_dir, f"bl_trial_{trial.number:05d}"))
        logger.info(f"BL TRIAL {trial.number} | tau={tau:.4f} ra={risk_aversion:.4f} "
                    f"omega={omega_scale:.4f} max_w={max_weight:.4f}")
        split_sharpes = []
        for split_i, (train_idx, val_idx) in enumerate(splits):
            try:
                bundle = _build_topn_ensemble_for_bl(
                    df=df, assets=assets, lstm_studies=lstm_studies,
                    split_train_idx=train_idx, split_val_idx=val_idx,
                    dates_index=dates_index, seed=seed, logger=logger, spreads=spreads,
                )
            except Exception as e:
                logger.info(f"[PRUNE] Ensemble failed on split {split_i}: {e}")
                raise optuna.TrialPruned()
            ensemble = bundle["ensemble_pred"]
            y_raw    = bundle["y_raw"]
            y_mask   = bundle["y_mask"]
            cov      = bundle["train_cov"]
            val_pred_mask = y_mask[val_idx] & np.isfinite(ensemble[val_idx])
            bl_res = run_black_litterman_backtest(
                dates_target=dates_index[val_idx + 1],
                y_true=y_raw[val_idx], y_pred=ensemble[val_idx],
                mask=val_pred_mask, asset_names=assets,
                train_cov=cov, tau=tau, risk_aversion=risk_aversion,
                omega_scale=omega_scale, long_only=long_only, max_weight=max_weight,
            )
            sharpe = bl_res["metrics"]["sharpe"]
            if not np.isfinite(sharpe): sharpe = -999.0
            split_sharpes.append(float(sharpe))
            logger.info(f"[BL SPLIT {split_i}] sharpe={sharpe:.4f}")
            trial.report(float(np.mean(split_sharpes)), step=split_i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        final = float(np.mean(split_sharpes))
        trial.set_user_attr("mean_val_sharpe", final)
        logger.info(f"BL TRIAL {trial.number} DONE | mean_sharpe={final:.4f}")
        return final

    bl_study = optuna.create_study(
        study_name="black_litterman_hpo",
        direction="maximize",
        storage=storage,
        load_if_exists=True,
        sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    bl_study.optimize(bl_objective, n_trials=n_trials_bl, gc_after_trial=True,
                      show_progress_bar=True)
    best = bl_study.best_trial
    with open(os.path.join(out_dir, "best_bl_trial.json"), "w") as fh:
        json.dump({"best_value": best.value, "best_params": best.params,
                   "n_trials": len(bl_study.trials)}, fh, indent=2)
    print(f"\n=== BEST BL TRIAL ===")
    print(f"Mean val Sharpe: {best.value:.4f}")
    print(f"Params: {best.params}")
    return bl_study

## 17) Final test — refit all LSTMs + BL on test set

In [ ]:
def run_final_test(
    df: pd.DataFrame,
    assets: List[str],
    lstm_studies: List[optuna.Study],
    bl_study: optuna.Study,
    out_dir: str,
    seed: int = SEED_PORTFOLIO_LSTM_RETURNS_V5_2_X_REDDIT_NEWS,
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """
    Final out-of-sample evaluation with mid-test refit.
    Block 1: train →2023-06-30, test 2023-07-01..2024-06-30
    Block 2: train →2024-06-30, test 2024-07-01..2025-06-30
    """
    safe_makedirs(os.path.join(out_dir, "final_test"))
    logger      = setup_logger(os.path.join(out_dir, "final_test"))
    dates_index = pd.DatetimeIndex(df.index)

    best_bl       = bl_study.best_trial
    tau           = best_bl.params["tau"]
    risk_aversion = best_bl.params["risk_aversion"]
    omega_scale   = best_bl.params["omega_scale"]
    max_weight    = best_bl.params["max_weight"]
    long_only     = True
    logger.info(f"BL params: tau={tau:.4f} ra={risk_aversion:.4f} "
                f"omega={omega_scale:.4f} max_w={max_weight:.4f}")

    blocks = [
        ("2023-06-30", "2023-07-01", "2024-06-30"),
        ("2024-06-30", "2024-07-01", "2025-06-30"),
    ]
    all_weights_list = []
    all_returns_list = []

    for block_i, (train_end, test_start, test_end) in enumerate(blocks):
        logger.info(f"=== BLOCK {block_i} | train→{train_end} "
                    f"test {test_start}..{test_end} ===")
        anchor_idx   = np.arange(len(dates_index) - 1, dtype=np.int64)
        target_dates = dates_index[1:]
        train_idx = anchor_idx[target_dates <= pd.Timestamp(train_end)]
        test_idx  = anchor_idx[
            (target_dates >= pd.Timestamp(test_start)) &
            (target_dates <= pd.Timestamp(test_end))
        ]
        T = len(df)
        N = len(assets)
        pred_full_list   = []
        canonical_y_raw  = None
        canonical_y_mask = None

        for rank, study in enumerate(lstm_studies):
            best = study.best_trial
            picks                 = best.user_attrs["tech_picks"]
            reddit_sentiment_cols = best.user_attrs["reddit_sentiment_cols"]
            news_sentiment_cols   = best.user_attrs["news_sentiment_cols"]
            reddit_nan_strategy   = best.user_attrs["reddit_nan_strategy"]
            news_nan_strategy     = best.user_attrs["news_nan_strategy"]
            lookback              = int(best.params["lookback"])
            model_params          = _build_model_params_from_trial(best)

            _, X_raw, y_model, y_raw, y_mask, feat_names, market_raw =                 build_raw_arrays_merged(
                    df=df, assets=assets, picks=picks, features=FEATURES,
                    logger=logger,
                    reddit_sentiment_cols=reddit_sentiment_cols,
                    news_sentiment_cols=news_sentiment_cols,
                    reddit_nan_strategy=reddit_nan_strategy,
                    news_nan_strategy=news_nan_strategy,
                    spreads=spreads,
                )
            if canonical_y_raw is None:
                canonical_y_raw  = y_raw.copy()
                canonical_y_mask = y_mask.copy()

            X_train_raw       = X_raw[train_idx]
            market_train_raw  = market_raw[train_idx] if market_raw is not None else None
            fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
            X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

            sample_valid = compute_sample_validity(
                X_trans, y_mask, lookback, min_valid_targets=max(1, N // 3))
            train_core_idx, es_idx = split_train_for_early_stop(
                train_idx, dates_index, es_years=1, min_train_days=252)
            train_core_t = train_core_idx[sample_valid[train_core_idx]]
            es_t         = es_idx[sample_valid[es_idx]]
            test_t       = test_idx[sample_valid[test_idx]]

            if len(train_core_t) < 64 or len(es_t) < 32:
                logger.warning(f"Rank {rank} block {block_i}: too few samples, skipping.")
                continue

            input_dim = X_trans.shape[2] + (1 if market_trans is not None else 0)
            model, _  = train_one_model(
                X=X_trans, y=y_model, y_mask=y_mask, market=market_trans,
                train_t=train_core_t, es_t=es_t, lookback=lookback,
                input_dim_per_asset=input_dim, output_dim=N,
                params=model_params, seed=seed, logger=logger,
            )
            test_eval = evaluate_model_on_t_indices(
                model=model, X=X_trans, y=y_model, y_mask=y_mask,
                market=market_trans, t_idx=test_t, lookback=lookback,
                batch_size=model_params["batch_size"],
                loss_name=model_params["loss_name"],
                huber_delta=model_params["huber_delta"],
            )
            pred_full = np.full((T, N), np.nan, dtype=np.float32)
            if len(test_eval["t_idx"]) > 0:
                pred_full[test_eval["t_idx"]] = test_eval["y_pred"]
            pred_full_list.append(pred_full)
            logger.info(f"Rank {rank} block {block_i}: "
                        f"ic_s={test_eval['metrics']['ic_spearman']:.4f}")

        if not pred_full_list:
            raise ValueError(f"Block {block_i}: no predictions produced.")

        ensemble  = nanmean_predictions(pred_full_list)
        train_cov = estimate_train_covariance(
            canonical_y_raw[train_idx], canonical_y_mask[train_idx])
        test_pred_mask = canonical_y_mask[test_idx] & np.isfinite(ensemble[test_idx])
        bl_res = run_black_litterman_backtest(
            dates_target=dates_index[test_idx + 1],
            y_true=canonical_y_raw[test_idx], y_pred=ensemble[test_idx],
            mask=test_pred_mask, asset_names=assets,
            train_cov=train_cov, tau=tau, risk_aversion=risk_aversion,
            omega_scale=omega_scale, long_only=long_only, max_weight=max_weight,
        )
        logger.info(f"Block {block_i} test: {bl_res['metrics']}")
        all_weights_list.append(bl_res["weights_df"])
        all_returns_list.append(bl_res["returns_df"])

    weights_df = pd.concat(all_weights_list).sort_index()
    returns_df = pd.concat(all_returns_list).sort_index()
    port_rets  = returns_df["portfolio_return"].values
    returns_df["portfolio_wealth"] = np.cumprod(1.0 + np.nan_to_num(port_rets, nan=0.0))

    final_metrics = portfolio_metrics_from_returns(port_rets[np.isfinite(port_rets)])
    final_metrics["avg_daily_turnover"] = float(
        np.abs(weights_df.diff().fillna(0.0)).sum(axis=1).mean())

    out = os.path.join(out_dir, "final_test")
    weights_df.to_csv(os.path.join(out, "test_bl_weights.csv"))
    returns_df.to_csv(os.path.join(out, "test_bl_returns.csv"))
    with open(os.path.join(out, "test_metrics.json"), "w") as fh:
        json.dump(final_metrics, fh, indent=2)

    print("\n=== FINAL TEST METRICS ===")
    for k, v in final_metrics.items():
        print(f"  {k:30s}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")
    print(f"Weights: {os.path.join(out, 'test_bl_weights.csv')}")
    print(f"Returns: {os.path.join(out, 'test_bl_returns.csv')}")
    return {"metrics": final_metrics, "weights_df": weights_df, "returns_df": returns_df}

## 18) Search space inspector  *(unchanged)*

In [ ]:
def estimate_search_space(study: optuna.Study) -> int:
    import math
    if not study.trials:
        raise ValueError("Run at least 1 trial first so distributions are populated.")
    distributions         = study.trials[0].distributions
    discrete_combinations = 1
    continuous_params     = []
    print("Search space:")
    for name, dist in distributions.items():
        if isinstance(dist, optuna.distributions.CategoricalDistribution):
            n = len(dist.choices)
            discrete_combinations *= n
            print(f"  {name:35s}  categorical  {n:>4d} choices")
        elif isinstance(dist, optuna.distributions.IntDistribution):
            n = (dist.high - dist.low) // dist.step + 1
            discrete_combinations *= n
            print(f"  {name:35s}  int          {n:>4d} levels  [{dist.low} .. {dist.high}]")
        elif isinstance(dist, optuna.distributions.FloatDistribution):
            continuous_params.append(name)
            scale = "log" if dist.log else "linear"
            print(f"  {name:35s}  float ({scale:6s})  [{dist.low:.2e} .. {dist.high:.2e}]  ← continuous")
    suggested = math.ceil(math.sqrt(discrete_combinations))
    print()
    print(f"Discrete combinations : {discrete_combinations}")
    print(f"Continuous params     : {continuous_params}")
    print(f"Suggested N_TRIALS    : ceil(sqrt({discrete_combinations})) = {suggested}")
    return suggested

## 19) Main execution

In [ ]:
# ─── Load data ───────────────────────────────────────────────────────────────
df = pd.read_parquet(DATASET_PATH)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

with open(SPREADS_PATH) as f:
    spreads_dict = json.load(f)
spreads = np.array([spreads_dict.get(a, 0.0) for a in ASSETS], dtype=np.float32)

print(f"Dataset: {df.shape}  ({df.index[0].date()} → {df.index[-1].date()})")
print(f"Assets:  {ASSETS}")
print(f"Spreads: {dict(zip(ASSETS, spreads.tolist()))}")

In [ ]:
# ─── Sanity-check that all merged sentiment columns exist in df ───────────────
missing_cols = []

for cfg in MERGED_CONFIGS:
    all_sents = cfg["reddit_sentiment_cols"] + cfg["news_sentiment_cols"]
    for a in ASSETS:
        for sf in all_sents:
            col = f"{a}_{sf}"
            if col not in df.columns:
                missing_cols.append(col)

for flag in ["_REDDIT_NAN", "_NEWS_NAN"]:
    if flag not in df.columns:
        missing_cols.append(flag)

if missing_cols:
    # Deduplicate
    missing_cols = list(dict.fromkeys(missing_cols))
    print(f"WARNING: {len(missing_cols)} expected columns not found in df.")
    print("First 10:", missing_cols[:10])
else:
    print("All merged sentiment columns present in dataset. ✓")

In [ ]:
# ─── Step 1: tune one merged LSTM per rank ───────────────────────────────────
# Creates {OUT_DIR}/lstm_top{i}/lstm_top{i}.db for each rank i.
# Only LSTM + training HPs are searched — all features are frozen from merged_configs.
# Set N_TRIALS in Config before running.

safe_makedirs(OUT_DIR)

lstm_studies = run_per_lstm_search(
    df=df,
    assets=ASSETS,
    merged_configs=MERGED_CONFIGS,
    out_dir=OUT_DIR,
    n_trials=N_TRIALS,
    seeds=MODEL_SEEDS,
    spreads=spreads,
)

print(f"\nCompleted {len(lstm_studies)} merged LSTM studies.")
for rank, s in enumerate(lstm_studies):
    best = s.best_trial
    print(f"  Rank {rank} | best merged IC={best.value:.5f}")

In [ ]:
# ─── (Optional) Inspect HP search space ──────────────────────────────────────
estimate_search_space(lstm_studies[0])
print(f"You did {len(lstm_studies[0].trials)} trials for rank-0 model.")

In [ ]:
# ─── Step 2: tune BL hyperparameters over the merged ensemble ────────────────
# top_n is fixed = TOP_N (already determined by v5.2.2 BL study).
# Searches: tau, risk_aversion, omega_scale, max_weight.

bl_study = run_bl_search(
    df=df,
    assets=ASSETS,
    lstm_studies=lstm_studies,
    out_dir=OUT_DIR,
    n_trials_bl=N_TRIALS_BL,
    seed=BL_SEED,
    spreads=spreads,
)

In [ ]:
# def delete_bl_best_trial(out_dir: str):
#     db_path  = os.path.join(out_dir, "black_litterman_optuna.db")
#     study    = optuna.load_study(study_name="black_litterman_hpo", storage=f"sqlite:///{db_path}")
#     trial_id = study.best_trial._trial_id

#     with sqlite3.connect(db_path) as con:
#         for table in ["trial_values", "trial_params", "trial_user_attributes",
#                       "trial_system_attributes", "trial_intermediate_values", "trials"]:
#             con.execute(f"DELETE FROM {table} WHERE trial_id = ?", (trial_id,))
#         con.commit()
#     print(f"BL | deleted trial_id={trial_id}")

# delete_bl_best_trial(OUT_DIR)

In [ ]:
estimate_search_space(bl_study)
print(f"You did {len(bl_study.trials)} trials for the BL model.")

In [ ]:
# ─── Step 3: final test with mid-test refit ───────────────────────────────────
# Refits all merged LSTMs on full train window, runs BL with best params,
# evaluates on out-of-sample test blocks 2023-H2 and 2024-H2.

results = run_final_test(
    df=df,
    assets=ASSETS,
    lstm_studies=lstm_studies,
    bl_study=bl_study,
    out_dir=OUT_DIR,
    seed=BL_SEED,
    spreads=spreads,
)

In [ ]:
weights = results["weights_df"].values

In [ ]:
import sys
from pathlib import Path
from src.metrics import PortfolioMetrics

pm = PortfolioMetrics()
print(pm.summary(weights))
pm.plot_weights(weights)
pm.plot_cumulative_returns(weights)

In [ ]:
np.save("LSTMvia-returns_reddit+news_weights.npy", weights)